In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import auc

sys.path.append(os.path.abspath("../"))

from util import dataset as u_dataset

In [ ]:
def read_csv_file(file_path):
    """
    Reads a CSV file from the given path and returns a pandas DataFrame.

    Parameters:
    file_path (str): Path to the CSV file.

    Returns:
    pd.DataFrame: DataFrame containing the CSV data.
    """
    try:
        df = pd.read_csv(file_path)
        return df
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None


classifier_basic = read_csv_file("../../data/evaluation/classifier-basic/run_1.csv")

ceilings = {
    u_dataset.IntersectionType.L: float(
        classifier_basic[classifier_basic["architecture"] == "conv_v1"][
            "recall_per_l_intersection"
        ].iloc[0]
    ),
    u_dataset.IntersectionType.T: float(
        classifier_basic[classifier_basic["architecture"] == "conv_v1"][
            "recall_per_t_intersection"
        ].iloc[0]
    ),
    u_dataset.IntersectionType.X: float(
        classifier_basic[classifier_basic["architecture"] == "conv_v1"][
            "recall_per_x_intersection"
        ].iloc[0]
    ),
}

In [ ]:
log_path = Path("../../logs/fit/")

In [ ]:
architectures = [
    r"$\mathrm{Cla}_1$",
    r"$\mathrm{Cla}_2$",
    r"$\mathrm{Cla}_3$",
    r"$\mathrm{Cla}_4$",
]

In [ ]:
# --- Constants ---
log_path = Path("../../logs/fit/")
ctx_sizes = ["0", "32"]
specifications = ctx_sizes
classes = list(u_dataset.IntersectionType)[1:]
runs = [1, 2, 3]
architectures = ["v1", "v2", "v3", "v4"]


# --- Data Extraction ---
def extract_ap_values():
    ap_values = {
        cls.name: [[[] for _ in range(len(specifications))] for _ in range(len(architectures))]
        for cls in list(u_dataset.IntersectionType)[1:]
    }
    ceiling_values = {
        cls.name: [[[] for _ in range(len(specifications))] for _ in range(len(architectures))]
        for cls in list(u_dataset.IntersectionType)[1:]
    }

    for run in runs:
        for arch_idx, arch in enumerate(architectures):
            for ctx_idx, ctx in enumerate(specifications):
                for cls in list(u_dataset.IntersectionType)[1:]:
                    if ctx == "0":
                        base_dir = Path(log_path, "classifier-basic", f"run_{run}", arch)
                    else:
                        base_dir = Path(log_path, "classifier-n_ctx", f"run_{run}", arch)
                    folders = [f for f in base_dir.iterdir() if f.is_dir()]
                    if len(folders) == 1:
                        path_to_model = folders[0]
                    else:
                        raise ValueError(
                            f"Expected exactly one folder in {base_dir}, found {len(folders)}"
                        )

                    # Load AP values for the class
                    precisions = np.load(
                        path_to_model.as_posix() + f"/precisions/intersections_{cls.name}.npy"
                    )
                    recalls = np.load(
                        path_to_model.as_posix() + f"/recalls/intersections_{cls.name}.npy"
                    )
                    ap_values[cls.name][arch_idx][ctx_idx].append(auc(recalls, precisions))
                    ceiling_values[cls.name][arch_idx][ctx_idx].append(np.max(recalls))

    # Compute ceiling means
    # ceiling_values_mean = {
    #     cls: np.max(ceiling_values[cls], axis=2)  # Max over runs and ctx
    #     for cls in classes
    # }
    return ap_values

In [ ]:
classes = ["L-Kreuzung", "T-Kreuzung", "X-Kreuzung"]


# --- Plotting ---
def plot_ap_values(ap_values):
    x = np.arange(len(ctx_sizes))
    n_classes = len(list(u_dataset.IntersectionType)[1:])
    bar_width = 0.22
    offsets = np.linspace(-(n_classes - 1) / 2, (n_classes - 1) / 2, n_classes) * bar_width
    colors = ["#56B4E9", "#D55E00", "#CC79A7"]  # Okabe-Ito

    os.makedirs("../../plots/ctx_saturation", exist_ok=True)

    # ── Helpers ────────────────────────────────────────────────────────────────────
    def style_ax(ax):
        ax.set_xticks(x)
        ax.set_xticklabels(ctx_sizes, fontsize=11)
        ax.set_xlabel(r"$n_{\mathrm{ctx}}$", fontsize=11)
        ax.yaxis.grid(True, linestyle="--", alpha=0.6, zorder=0)
        ax.set_axisbelow(True)
        ax.spines[["top", "right"]].set_visible(False)

    def draw_ceilings(ax, ceiling_dict):
        for cls, color in zip(list(u_dataset.IntersectionType)[1:], colors, strict=True):
            ax.axhline(
                y=ceiling_dict[cls],
                color=color,
                linewidth=1.2,
                linestyle="--",
                alpha=0.8,
                zorder=2,
            )

    def draw_points_and_mean(ax, data_dict, arch_idx):
        for cls, color, offset in zip(
            list(u_dataset.IntersectionType)[1:], colors, offsets, strict=True
        ):
            means = []
            for i, samples in enumerate(data_dict[cls.name][arch_idx]):
                samples = np.array(samples)
                mean = np.mean(samples)
                means.append(mean)

                # Jittered scatter points
                jitter = np.linspace(-0.06, 0.06, len(samples))
                ax.scatter(
                    x[i] + offset + jitter,
                    samples,
                    color=color,
                    s=30,
                    zorder=5,
                    alpha=0.7,
                    edgecolors="white",
                    linewidths=0.4,
                )
                # Mean hline
                ax.hlines(
                    mean,
                    x[i] + offset - 0.09,
                    x[i] + offset + 0.09,
                    colors=color,
                    linewidth=2.5,
                    zorder=6,
                )

            # Connecting line between the ctx means
            ax.plot(
                x + offset,
                means,
                color=color,
                linewidth=0.7,
                linestyle="-",
                zorder=4,
                alpha=0.9,
            )

    def add_legend(ax, with_ceiling=False):
        handles = [
            plt.Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor=color,
                markersize=7,
                label=cls,
            )
            for cls, color in zip(classes, colors, strict=True)
        ]
        if with_ceiling:
            handles += [
                plt.Line2D(
                    [0],
                    [0],
                    color=color,
                    linewidth=1.2,
                    linestyle="--",
                    alpha=0.8,
                    label=r"Recall$@K_c$ für " + f"{cls}",
                )
                for cls, color in zip(classes, colors, strict=True)
            ]
        ax.legend(
            handles=handles,
            loc="lower right",
            framealpha=0.9,
            fontsize=9,
            ncols=2 if with_ceiling else 1,
        )

    # ── Plot for each architecture ────────────────────────────────────────────────
    for arch_idx, arch in enumerate(architectures):
        fig, ax = plt.subplots(figsize=(7, 5))
        draw_points_and_mean(ax, ap_values, arch_idx)
        draw_ceilings(ax, ceilings)
        ax.set_ylim(0.4, 1.05)
        style_ax(ax)
        ax.set_ylabel("AP", fontsize=11)
        add_legend(ax, with_ceiling=True)
        plt.tight_layout()
        plt.savefig(f"../../plots/n_ctx/conv_{arch}_intersection_APs.pdf")
        plt.show()


# --- Execute ---
ap_values = extract_ap_values()
plot_ap_values(ap_values)